# 04c — Privacidad y PII

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/bloque4/04c-privacidad.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> Este tema forma parte del [Bloque 4 — Gobernanza](04-gobernanza.qmd).
> Se recomienda haber leído antes [03a — Datos
> Sintéticos](../bloque3/03a-datos-sinteticos.qmd), donde se introducen
> SDV, CTGAN y TVAE.

## 3. Privacidad y PII

Antes de enviar cualquier dato a un LLM (incluso local) necesitas
detectar y enmascarar la información personal identificable (PII). Y si
vas a generar datos sintéticos, debes verificar que no se pueda
re-identificar a los registros originales.

### 3.1 Detección de PII con regex

``` python
#| echo: true
#| eval: true

import re

PATRONES_PII = {
    "email":       r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b",
    "telefono":    r"\b(?:\+34|0034|34)?[ -]?[6789]\d{2}[ -]?\d{2}[ -]?\d{2}[ -]?\d{2}\b",
    "dni":         r"\b\d{8}[ -]?[A-Z]\b",
    "nie":         r"\b[XYZ]\d{7}[ -]?[A-Z]\b",
    "tarjeta":     r"\b(?:\d[ -]?){13,19}\b",
    "iban":        r"\bES\d{2}[ ]?\d{4}[ ]?\d{4}[ ]?\d{4}[ ]?\d{4}[ ]?\d{4}\b",
    "cp":          r"\b(?:28|41|29|46|08|30|07|39|48|20|47)\d{4}\b",
}

def detectar_pii(texto: str) -> dict:
    """Devuelve un dict con las PII encontradas por tipo."""
    encontrados = {}
    for tipo, patron in PATRONES_PII.items():
        matches = re.findall(patron, texto)
        if matches:
            encontrados[tipo] = list(set(matches))
    return encontrados

def enmascarar_pii(texto: str) -> str:
    """Sustituye cada PII por su tipo entre corchetes."""
    enmascarado = texto
    for tipo, patron in PATRONES_PII.items():
        enmascarado = re.sub(patron, f"[{tipo.upper()}]", enmascarado)
    return enmascarado


# Demo
texto = """
Cliente: Juan Pérez (juan.perez@empresa.com, tel. 612 345 678).
DNI: 12345678Z. Tarjeta: 4532 1234 5678 9012. CP: 41001.
"""

pii = detectar_pii(texto)
print("PII detectada:")
for tipo, vals in pii.items():
    print(f"  {tipo:10s} → {vals}")

print(f"\nTexto enmascarado:\n{enmascarar_pii(texto).strip()}")
```

> **Regex no es suficiente**
>
> Estas regex atrapan formatos comunes, pero fallan con: - Nombres
> completos (necesitas NER o un LLM). - Direcciones físicas completas. -
> Variantes con espacios, puntos o caracteres Unicode.
>
> Para producción usa herramientas especializadas: [Microsoft
> Presidio](https://github.com/microsoft/presidio) (Microsoft, 2025) o
> [comprehend](https://aws.amazon.com/comprehend/) de AWS.

### 3.2 Datos sintéticos y privacidad

Un error frecuente: *“Si genero datos sintéticos, ya no hay riesgo de
privacidad”*. Falso. Los modelos generativos pueden memorizar registros
del conjunto de entrenamiento y emitirlos casi literalmente.

``` python
#| echo: true
#| eval: true

import numpy as np
from sklearn.preprocessing import StandardScaler

def distancia_al_mas_cercano(df_sint: np.ndarray, df_real: np.ndarray) -> dict:
    """
    Distancia al registro real más cercano (DCR) sobre columnas numéricas.
    - DCR cercano a 0 → registro sintético casi idéntico a uno real (riesgo alto).
    - DCR grande → registro sintético está lejos de cualquier real (más seguro).
    """
    d_min, d_media = [], []
    for fila in df_sint:
        dists = np.linalg.norm(df_real - fila, axis=1)
        d_min.append(dists.min())
        d_media.append(dists.mean())
    return {
        "dcr_min":  float(np.min(d_min)),
        "dcr_p5":   float(np.percentile(d_min, 5)),
        "dcr_mean": float(np.mean(d_min)),
        "dcr_std":  float(np.std(d_min)),
    }


# Mini-demo: comparar datos casi idénticos vs. datos con ruido
np.random.seed(42)
df_real = np.random.randn(500, 4)
df_copia       = df_real + np.random.randn(500, 4) * 1e-3   # casi idéntica
df_independiente = np.random.randn(500, 4) * 2               # distribución similar, no copia

print("Dataset COPIA (riesgo de re-identificación alto):")
for k, v in distancia_al_mas_cercano(df_copia, df_real).items():
    print(f"  {k:10s} = {v:.4f}")

print("\nDataset INDEPENDIENTE (mismo tipo de datos, sin copia):")
for k, v in distancia_al_mas_cercano(df_independiente, df_real).items():
    print(f"  {k:10s} = {v:.4f}")
```

### 3.3 Privacidad diferencial con SDV

SDV permite configurar un presupuesto de privacidad `epsilon`: cuanto
menor, más ruido se añade y más seguros son los datos sintéticos
resultantes, pero menos útiles.

``` python
#| echo: true
#| eval: false

import pandas as pd
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

df = pd.read_csv("tu_dataset_corporativo.csv")
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)

# Sin privacidad
synth_normal = CTGANSynthesizer(metadata, epochs=100)
synth_normal.fit(df)
df_normal = synth_normal.sample(1000)

# Con privacidad diferencial (epsilon=1.0 = privacidad fuerte, menor utilidad)
synth_dp = CTGANSynthesizer(metadata, epochs=100, anonymization_config={"epsilon": 1.0})
synth_dp.fit(df)
df_dp = synth_dp.sample(1000)
```

> **Buenas prácticas con datos sintéticos**
>
> - [ ] Calcular DCR (media y percentil 5) sobre datos sintéticos
>   vs. reales.
> - [ ] Si `dcr_p5 < 0.01` (registros casi idénticos) → riesgo alto, no
>   publiques.
> - [ ] Configurar `epsilon` entre 1.0 (privacidad fuerte) y 10.0
>   (utilidad alta).
> - [ ] Documentar el esquema y los parámetros en la Model Card del
>   dataset sintético (Mitchell et al., 2019).
> - [ ] Considerar ataques de membership inference como test
>   adversarial. Para privacidad diferencial formal, véase Dwork & Roth
>   (2014).

# References

Dwork, C., & Roth, A. (2014). *The algorithmic foundations of
differential privacy* (Vol. 9, pp. 211–407). Foundations; Trends® in
Theoretical Computer Science. <https://doi.org/10.1561/0400000042>

Microsoft. (2025). *Presidio – PII detection and anonymization*.
Repositorio de código. <https://github.com/microsoft/presidio>

Mitchell, M., Wu, S., Zaldivar, A., Barnes, P., Vasserman, L.,
Hutchinson, B., Spitzer, E., Raji, I. D., & Gebru, T. (2019). Model
cards for model reporting. *Proceedings of the 2019 Conference on
Fairness, Accountability, and Transparency (FAT\*)*, 220–229.
<https://doi.org/10.1145/3287560.3287596>